# Object Detection Training — Multi-Dataset, Accuracy-First

Two things changed from the first pass of this notebook:

1. **Not locked to Raspberry Pi 3 constraints anymore.** We're training for accuracy first (this model
   is the foundation you'll later combine with SLAM), so default `imgsz` is back up to 640 and the
   nano-only restriction is gone. Deployment-target export (Section 6) is now optional/later — pick it
   back up once accuracy on your own data is solid.
2. **Built to take multiple datasets.** You'll be adding more over time (starting with `streetscape-waytm`
   from Roboflow). Section 2 merges any number of YOLO-format datasets — even ones with different class
   lists — into one combined training set, instead of the notebook only handling a single Roboflow export.

## Model: YOLO26 instead of YOLOv8

Ultralytics released **YOLO26** in January 2026 — it's the current recommended model for edge/embedded
deployment specifically (which matters once you're back to targeting a Pi/Jetson): natively NMS-free,
and Ultralytics reports up to ~43% faster CPU inference than YOLO11 at equal or better accuracy, with a
lighter detection head that's easier for low-power accelerators to run.

There's also **YOLOv12** (attention-based, from the research community) — it can edge out YOLO26 on raw
COCO accuracy in some independent benchmarks, but Ultralytics itself now positions v12 as a
research/benchmarking line rather than a deployment target, and it's slower on CPU / more memory-hungry,
which works against you on constrained hardware. For this project — edge deployment, power-constrained —
**YOLO26n** is the better default. The API is identical to YOLOv8 (`YOLO("yolo26n.pt")` instead of
`YOLO("yolov8n.pt")`), so nothing else in this pipeline changes.


## 1. Environment setup

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q ultralytics roboflow opencv-python-headless
import ultralytics
ultralytics.checks()


## 2. Multi-dataset merge

Add every Roboflow project you want included to `ROBOFLOW_DATASETS` below (workspace, project slug,
version). Each one is downloaded in YOLOv8 format, then `merge_datasets.py` combines them into one
dataset — building a unified class list across all of them (so `streetscape-waytm`'s classes and any
future project's classes don't collide or silently overwrite each other) and remapping every label file
to the merged class indices.

This cell writes `merge_datasets.py` into the Colab environment. It's also included in your file bundle
so you can drop it straight into the repo and reuse it outside Colab.


In [ ]:
%%writefile merge_datasets.py
"""
Merge any number of YOLO-format (Roboflow-exported) detection datasets into one combined dataset,
building a unified class list even when the input datasets have different classes.

Usage:
    python3 merge_datasets.py --datasets DATASET_DIR_1 DATASET_DIR_2 ... --out MERGED_DIR

Each DATASET_DIR is expected to look like a standard Roboflow YOLOv8 export:
    DATASET_DIR/
      data.yaml        # contains a `names:` list
      train/images, train/labels
      valid/images, valid/labels
      test/images,  test/labels   (optional)
"""
import argparse
import os
import shutil
import yaml


def load_class_names(dataset_dir):
    with open(os.path.join(dataset_dir, "data.yaml"), "r") as f:
        data = yaml.safe_load(f)
    return list(data["names"])


def build_global_classes(dataset_dirs):
    global_classes = []
    per_dataset_names = []
    for d in dataset_dirs:
        names = load_class_names(d)
        per_dataset_names.append(names)
        for n in names:
            if n not in global_classes:
                global_classes.append(n)
    return global_classes, per_dataset_names


def remap_and_copy_split(dataset_dir, split, local_names, global_classes, out_dir, tag):
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    if not os.path.isdir(img_dir):
        return 0

    out_img_dir = os.path.join(out_dir, split, "images")
    out_lbl_dir = os.path.join(out_dir, split, "labels")
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    remap = {i: global_classes.index(name) for i, name in enumerate(local_names)}
    count = 0

    for fname in os.listdir(img_dir):
        stem, ext = os.path.splitext(fname)
        new_stem = f"{tag}_{stem}"

        shutil.copy2(os.path.join(img_dir, fname), os.path.join(out_img_dir, new_stem + ext))

        src_lbl = os.path.join(lbl_dir, stem + ".txt")
        dst_lbl = os.path.join(out_lbl_dir, new_stem + ".txt")
        if os.path.exists(src_lbl):
            with open(src_lbl, "r") as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                old_cls = int(parts[0])
                parts[0] = str(remap[old_cls])
                new_lines.append(" ".join(parts))
            with open(dst_lbl, "w") as f:
                f.write("\n".join(new_lines) + ("\n" if new_lines else ""))
        else:
            # negative example (no objects) — still valid, write empty label file
            open(dst_lbl, "w").close()

        count += 1

    return count


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--datasets", nargs="+", required=True)
    parser.add_argument("--out", required=True)
    args = parser.parse_args()

    global_classes, per_dataset_names = build_global_classes(args.datasets)
    print(f"Merged class list ({len(global_classes)}): {global_classes}")

    for dataset_dir, local_names in zip(args.datasets, per_dataset_names):
        tag = os.path.basename(os.path.normpath(dataset_dir))
        for split in ("train", "valid", "test"):
            n = remap_and_copy_split(dataset_dir, split, local_names, global_classes, args.out, tag)
            if n:
                print(f"  {dataset_dir} [{split}]: {n} images -> merged/{split}")

    merged_yaml = {
        "path": os.path.abspath(args.out),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(global_classes),
        "names": global_classes,
    }
    with open(os.path.join(args.out, "data.yaml"), "w") as f:
        yaml.safe_dump(merged_yaml, f, sort_keys=False)

    print(f"\nWrote merged dataset to {args.out}/data.yaml")


if __name__ == "__main__":
    main()


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")

# Add one entry per Roboflow project you want folded into training.
ROBOFLOW_DATASETS = [
    {"workspace": "soham-ojha", "project": "streetscape-waytm", "version": 1},
    # {"workspace": "...", "project": "...", "version": 1},
]

downloaded_dirs = []
for spec in ROBOFLOW_DATASETS:
    project = rf.workspace(spec["workspace"]).project(spec["project"])
    version = project.version(spec["version"])
    dataset = version.download("yolov8")
    downloaded_dirs.append(dataset.location)
    print(dataset.location)


In [ ]:
!python3 merge_datasets.py --datasets {' '.join(downloaded_dirs)} --out /content/merged_dataset


## 3. Train

`imgsz=640` is the accuracy-first default now — raise/lower it later once you're optimizing for a
specific deployment target. Swap `MODEL` to `"yolov8n.pt"` if you want a direct comparison against the
previous baseline.


In [ ]:
from ultralytics import YOLO

MODEL = "yolo26n.pt"

model = YOLO(MODEL)

results = model.train(
    data="/content/merged_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="uav_multi_dataset",
    name=MODEL.replace(".pt", ""),
)


## 4. Validate

In [ ]:
metrics = model.val()
print(metrics.box.map)     # mAP50-95
print(metrics.box.map50)   # mAP50


## 5. Inspect predictions

Worth doing before anything else — a quick visual sanity check on validation images catches labeling
or class-merge mistakes that mAP alone can hide.


In [ ]:
results = model.predict(source="/content/merged_dataset/valid/images", conf=0.4, save=True)
print(f"Predictions saved to {results[0].save_dir}")


## 6. Edge export (optional — revisit once accuracy is solid)

This section is deployment-target-specific, so don't run it until you've picked the hardware. Once you
have (Jetson vs. Pi 5 + Hailo vs. Coral, etc.), the export format differs:

- **Jetson (Orin Nano, etc.):** export `format="engine"` (TensorRT) for best throughput
- **Raspberry Pi 5 + Hailo-8L:** export to ONNX, then run it through the Hailo Dataflow Compiler (outside
  Colab — Hailo's toolchain isn't pip-installable here)
- **Coral Edge TPU (any host, incl. Pi 3):** export `format="tflite", int8=True`, then compile with
  `edgetpu_compiler` — this is what the earlier Pi 3 notebook did

Example (TFLite/Coral path, same as before):


In [ ]:
# model.export(format="tflite", int8=True, imgsz=640, data="/content/merged_dataset/data.yaml")
